# Exploration et visualisation du Registre Parcellaire Graphique (RPG 2025)

**Projet Python pour la Data Science (ENSAE) — *Eau et agriculture : qualité des eaux et usage des pesticides***

---

## Objectif de ce notebook

Dans le cadre de notre projet étudiant croisant **qualité des eaux (Hub'Eau)**, **achats de produits phytosanitaires (BNV-D)** et **occupation agricole des sols (RPG)**, ce notebook réalise l'exploration statistique et cartographique approfondie du **Registre Parcellaire Graphique (RPG — millésime 2025)** extrait via le service WFS de la Géoplateforme IGN (`data/processed/rpg/rpg2025/rpg_2025_*.parquet`).

L'analyse se concentre sur les cinq variables agronomiques d'intérêt :
- **`surface_ha`** : surface graphique déclarée de la parcelle (en hectares) ;
- **`libelle_categorie`** : grande catégorie d'occupation du sol (*Terres arables*, *Prairies permanentes*, *Cultures permanentes*, *Surfaces boisées*) ;
- **`libelle_groupe`** : groupe de cultures RPG (23 libellés regroupant les filières agricoles) ;
- **`libelle_culture`** : nature détaillée de la culture principale (144 cultures distinctes) ;
- **`bio`** : conduite de la parcelle en **Agriculture Biologique** (ou en conversion).

### Pourquoi cette exploration est-elle déterminante avant la fusion des trois bases ?
1. **Identifier l'exposition théorique aux pesticides par filière** : les intrants phytosanitaires sont massivement concentrés sur les **terres arables** (céréales à paille, maïs, colza, betterave, pomme de terre) et les **cultures permanentes** (vignes, vergers), tandis que les **prairies permanentes** et **estives** reçoivent très peu de traitements chimiques.
2. **Quantifier l'effet protecteur potentiel du Bio (`bio = True`)** : le cahier des charges de l'agriculture biologique interdisant les pesticides de synthèse, la part départementale et régionale de SAU en bio constitue une variable de contrôle majeure pour expliquer les écarts de contamination des cours d'eau et des nappes phréatiques.
3. **Caractériser la structure du parcellaire (`surface_ha`)** : l'opposition entre grands bassins d'openfield céréalier (parcelles vastes) et zones de polyculture-élevage ou de viticulture (parcellaire morcelé) éclaire la géographie agricole française.

In [ ]:
import os
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import Image, display

# Positionnement automatique à la racine du dépôt (quel que soit le sous-dossier d'exécution, ex. exploration/)
for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "data" / "processed" / "rpg").exists():
        os.chdir(_parent)
        break

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--",
    "font.size": 10.5,
})

# Table de correspondance Département -> Région administrative (INSEE)
DEPARTEMENT_VERS_REGION = {
    # Auvergne-Rhône-Alpes
    "01": "Auvergne-Rhône-Alpes", "03": "Auvergne-Rhône-Alpes", "07": "Auvergne-Rhône-Alpes",
    "15": "Auvergne-Rhône-Alpes", "26": "Auvergne-Rhône-Alpes", "38": "Auvergne-Rhône-Alpes",
    "42": "Auvergne-Rhône-Alpes", "43": "Auvergne-Rhône-Alpes", "63": "Auvergne-Rhône-Alpes",
    "69": "Auvergne-Rhône-Alpes", "73": "Auvergne-Rhône-Alpes", "74": "Auvergne-Rhône-Alpes",
    # Bourgogne-Franche-Comté
    "21": "Bourgogne-Franche-Comté", "25": "Bourgogne-Franche-Comté", "39": "Bourgogne-Franche-Comté",
    "58": "Bourgogne-Franche-Comté", "70": "Bourgogne-Franche-Comté", "71": "Bourgogne-Franche-Comté",
    "89": "Bourgogne-Franche-Comté", "90": "Bourgogne-Franche-Comté",
    # Bretagne
    "22": "Bretagne", "29": "Bretagne", "35": "Bretagne", "56": "Bretagne",
    # Centre-Val de Loire
    "18": "Centre-Val de Loire", "28": "Centre-Val de Loire", "36": "Centre-Val de Loire",
    "37": "Centre-Val de Loire", "41": "Centre-Val de Loire", "45": "Centre-Val de Loire",
    # Corse
    "2A": "Corse", "2B": "Corse",
    # Grand Est
    "08": "Grand Est", "10": "Grand Est", "51": "Grand Est", "52": "Grand Est",
    "54": "Grand Est", "55": "Grand Est", "57": "Grand Est", "67": "Grand Est",
    "68": "Grand Est", "88": "Grand Est",
    # Hauts-de-France
    "02": "Hauts-de-France", "59": "Hauts-de-France", "60": "Hauts-de-France",
    "62": "Hauts-de-France", "80": "Hauts-de-France",
    # Île-de-France
    "75": "Île-de-France", "77": "Île-de-France", "78": "Île-de-France", "91": "Île-de-France",
    "92": "Île-de-France", "93": "Île-de-France", "94": "Île-de-France", "95": "Île-de-France",
    # Normandie
    "14": "Normandie", "27": "Normandie", "50": "Normandie", "61": "Normandie", "76": "Normandie",
    # Nouvelle-Aquitaine
    "16": "Nouvelle-Aquitaine", "17": "Nouvelle-Aquitaine", "19": "Nouvelle-Aquitaine",
    "23": "Nouvelle-Aquitaine", "24": "Nouvelle-Aquitaine", "33": "Nouvelle-Aquitaine",
    "40": "Nouvelle-Aquitaine", "47": "Nouvelle-Aquitaine", "64": "Nouvelle-Aquitaine",
    "79": "Nouvelle-Aquitaine", "86": "Nouvelle-Aquitaine", "87": "Nouvelle-Aquitaine",
    # Occitanie
    "09": "Occitanie", "11": "Occitanie", "12": "Occitanie", "30": "Occitanie",
    "31": "Occitanie", "32": "Occitanie", "34": "Occitanie", "46": "Occitanie",
    "48": "Occitanie", "65": "Occitanie", "66": "Occitanie", "81": "Occitanie", "82": "Occitanie",
    # Pays de la Loire
    "44": "Pays de la Loire", "49": "Pays de la Loire", "53": "Pays de la Loire",
    "72": "Pays de la Loire", "85": "Pays de la Loire",
    # Provence-Alpes-Côte d'Azur
    "04": "Provence-Alpes-Côte d'Azur", "05": "Provence-Alpes-Côte d'Azur",
    "06": "Provence-Alpes-Côte d'Azur", "13": "Provence-Alpes-Côte d'Azur",
    "83": "Provence-Alpes-Côte d'Azur", "84": "Provence-Alpes-Côte d'Azur",
    # Outre-Mer (DROM)
    "971": "Outre-Mer (DROM)", "972": "Outre-Mer (DROM)", "973": "Outre-Mer (DROM)",
    "974": "Outre-Mer (DROM)", "976": "Outre-Mer (DROM)",
}

# Chargement rapide (sans la colonne geometry) des 101 fichiers Parquet de data/processed/rpg/rpg2025
t0 = time.time()
dossier_rpg = Path("data/processed/rpg/rpg2025")
if not dossier_rpg.exists():
    dossier_rpg = Path("data/processed/rpg")
fichiers_parquet = sorted(dossier_rpg.glob("rpg_2025_*.parquet"))

colonnes_utiles = [
    "id_parcelle", "annee", "code_dep", "nom_dep", "surface_ha",
    "categorie_culture", "libelle_categorie",
    "code_groupe", "libelle_groupe",
    "code_culture", "libelle_culture",
    "bio",
]

tables = [pq.read_table(f, columns=colonnes_utiles) for f in fichiers_parquet]
df = pa.concat_tables(tables).to_pandas()

# Bilan qualité avant création des colonnes dérivées
bilan_colonnes = pd.DataFrame({
    "Type": df[colonnes_utiles].dtypes.astype(str),
    "Nb valeurs manquantes": df[colonnes_utiles].isna().sum(),
    "% manquantes": (100 * df[colonnes_utiles].isna().mean()).round(2),
    "Nb modalités uniques": [df[c].nunique(dropna=True) for c in colonnes_utiles],
})

# Traitement explicite des valeurs manquantes sur libelle_categorie (codes "NA" ou NaN -> surfaces non attribuées / IAE)
df["libelle_categorie_propre"] = df["libelle_categorie"].fillna("Non attribué / Hors production (IAE, SNE)")
df["region"] = df["code_dep"].map(DEPARTEMENT_VERS_REGION).fillna("Autre")
df["bio"] = df["bio"].fillna(False).astype(bool)
df["surface_bio_ha"] = np.where(df["bio"], df["surface_ha"], 0.0)

duree = time.time() - t0
print(f"Chargement terminé en {duree:.2f} s : {len(df):,} parcelles réparties sur {len(fichiers_parquet)} départements.")
print(f"Surface totale déclarée (SAU) : {df['surface_ha'].sum():,.1f} ha ({df['surface_ha'].sum() / 1e6:.2f} millions d'ha)")
print(f"Surface conduite en Agriculture Biologique : {df['surface_bio_ha'].sum():,.1f} ha ({100 * df['surface_bio_ha'].sum() / df['surface_ha'].sum():.2f} % de la SAU)")

display(bilan_colonnes)

### Analyse de la structure et de la qualité des données

L'inspection exhaustive des **101 fichiers départementaux** (96 départements métropolitains et 5 DROM) met en évidence plusieurs constats méthodologiques importants :

1. **Couverture et volumétrie nationale** :
   - La base rassemble **9 776 236 parcelles** pour une surface agricole déclarée totale de **27 924 773,8 ha** (~**27,92 millions d'hectares**), ce qui correspond exactement à la Surface Agricole Utile (SAU) déclarée à la PAC en France.
   - À l'échelle nationale, **1 115 909 parcelles** (**11,41 %** des parcelles) sont conduites en **Agriculture Biologique (`bio = True`)**, représentant **2 899 677,4 ha**, soit **10,38 % de la SAU totale**.

2. **Complétude des variables d'intérêt et traitement des valeurs manquantes** :
   - Les variables **`surface_ha`**, **`libelle_groupe`** (23 modalités), **`libelle_culture`** (144 modalités) et **`bio`** ne présentent **aucune valeur manquante (`0,00 %`)**.
   - Seule la variable **`libelle_categorie`** comporte **992 783 valeurs manquantes (`10,16 %` des parcelles)**, qui se décomposent en :
     - **460 102 parcelles** ayant le code WFS `"NA"` (*Non attribué*, principalement des surfaces temporairement non admissibles `SNE`, bandes tampons `BTA` ou bordures de champs `BOR`) pour `185 468 ha` ;
     - **532 681 parcelles** sans code de catégorie (`NaN`) pour `84 363 ha` (micro-parcelles et infrastructures agro-écologiques d'une surface moyenne de seulement `0,16 ha`).
   - Conformément aux bonnes pratiques de nettoyage, nous ne supprimons pas ces lignes : nous les regroupons dans une modalité explicite **`"Non attribué / Hors production (IAE, SNE)"`** (qui ne pèse que **0,97 % de la surface totale**, soit `269 831 ha`).

---

## 1. Grandes catégories d'occupation du sol (`libelle_categorie`) et morphologie des parcelles (`surface_ha`)

Nous examinons d'abord comment se répartit l'espace agricole français entre les grandes catégories agronomiques (*Terres arables*, *Prairies permanentes*, *Cultures permanentes*, *Surfaces boisées*) et comment varie la **taille des parcelles (`surface_ha`)** selon la catégorie et le mode de production (`bio`).

In [ ]:
# Statistiques descriptives de surface_ha par grande catégorie de culture
stats_cat = (
    df.groupby("libelle_categorie_propre", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        moyenne_ha=("surface_ha", "mean"),
        q25_ha=("surface_ha", lambda s: s.quantile(0.25)),
        mediane_ha=("surface_ha", "median"),
        q75_ha=("surface_ha", lambda s: s.quantile(0.75)),
        q95_ha=("surface_ha", lambda s: s.quantile(0.95)),
        max_ha=("surface_ha", "max"),
        surface_bio_ha=("surface_bio_ha", "sum"),
    )
    .sort_values("surface_totale_ha", ascending=False)
)
stats_cat["part_surface_%"] = (100 * stats_cat["surface_totale_ha"] / df["surface_ha"].sum()).round(2)
stats_cat["part_parcelles_%"] = (100 * stats_cat["nb_parcelles"] / len(df)).round(2)
stats_cat["taux_bio_%"] = (100 * stats_cat["surface_bio_ha"] / stats_cat["surface_totale_ha"]).round(2)

display(
    stats_cat[[
        "libelle_categorie_propre", "nb_parcelles", "part_parcelles_%",
        "surface_totale_ha", "part_surface_%", "taux_bio_%",
        "moyenne_ha", "q25_ha", "mediane_ha", "q75_ha", "q95_ha", "max_ha"
    ]].round(2)
)

# Visualisation 1 : Poids des grandes catégories (gauche) & distribution de surface_ha (droite)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16.5, 6.4))

# Sous-graphique 1 : Part des surfaces vs Part du nombre de parcelles
cats_plot = stats_cat[stats_cat["libelle_categorie_propre"] != "Surfaces boisées"].copy()
y_pos = np.arange(len(cats_plot))
h_bar = 0.36

b1 = ax1.barh(
    y_pos - h_bar / 2,
    cats_plot["part_surface_%"],
    height=h_bar,
    color="#2b8cbe",
    label="Part de la surface totale (% SAU)",
)
b2 = ax1.barh(
    y_pos + h_bar / 2,
    cats_plot["part_parcelles_%"],
    height=h_bar,
    color="#fd8d3c",
    label="Part du nombre de parcelles (%)",
)

ax1.set_yticks(y_pos)
ax1.set_yticklabels(cats_plot["libelle_categorie_propre"], fontweight="bold")
ax1.invert_yaxis()
ax1.set_xlabel("Pourcentage du total national (%)")
ax1.set_xlim(0, 72)
ax1.set_title("Grandes catégories : Surface (% SAU)\nvs Nombre de parcelles (%)", fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True, facecolor="white")

for bar in b1:
    w = bar.get_width()
    ax1.text(w + 0.8, bar.get_y() + bar.get_height() / 2, f"{w:.1f} %", va="center", fontsize=9.5, color="#1b5e85", fontweight="bold")
for bar in b2:
    w = bar.get_width()
    ax1.text(w + 0.8, bar.get_y() + bar.get_height() / 2, f"{w:.1f} %", va="center", fontsize=9.5, color="#b85814")

# Sous-graphique 2 : Distribution de la taille des parcelles (échelle log10) par catégorie et statut Bio
categories_principales = ["Terres arables", "Prairies permanentes", "Cultures permanentes"]
couleurs_cat = {"Terres arables": "#e34a33", "Prairies permanentes": "#31a354", "Cultures permanentes": "#756bb1"}

donnees_box = []
labels_box = []
couleurs_box = []

for cat in categories_principales:
    for statut_bio, suffixe in [(False, "Conv."), (True, "Bio")]:
        vals = df.loc[
            (df["libelle_categorie_propre"] == cat) & (df["bio"] == statut_bio) & (df["surface_ha"] > 0),
            "surface_ha",
        ]
        donnees_box.append(vals.to_numpy())
        labels_box.append(f"{cat}\n({suffixe})")
        couleurs_box.append(couleurs_cat[cat])

bp = ax2.boxplot(
    donnees_box,
    orientation="horizontal",
    patch_artist=True,
    showfliers=False,
    whis=(10, 90),
    medianprops={"color": "#111111", "linewidth": 2.0},
)

for idx, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(couleurs_box[idx])
    patch.set_alpha(0.85 if idx % 2 == 1 else 0.45)
    if idx % 2 == 1:
        patch.set_hatch("//")

ax2.set_yticks(range(1, len(labels_box) + 1))
ax2.set_yticklabels(labels_box, fontsize=9.5)
ax2.invert_yaxis()
ax2.set_xscale("log")
ax2.set_xlabel("Surface de la parcelle en ha (échelle log10 — boîte : Q25–Q75, moustaches : D1–D9)")
ax2.set_title("Morphologie parcellaire (`surface_ha`)\npar catégorie et mode (Conv. vs Bio)", fontweight="bold", pad=12)

plt.tight_layout()
plt.show()

### Interprétation de la répartition par grande catégorie et de la taille des parcelles

Le tableau et les deux graphiques ci-dessus mettent en lumière trois faits structurels majeurs :

1. **Prédominance des Terres arables et des Prairies permanentes dans la SAU** :
   - Les **Terres arables (`TA`)** couvrent **16,68 millions d'hectares**, soit **59,73 % de la surface agricole française** (pour **47,63 %** des parcelles). C'est sur ce compartiment — soumis aux rotations annuelles et aux traitements herbicides/fongicides/insecticides — que se concentre l'essentiel des achats de substances actives BNV-D.
   - Les **Prairies permanentes (`PP`)** représentent le deuxième grand bloc avec **10,08 millions d'hectares** (**36,10 % de la SAU**, **34,39 %** des parcelles), jouant un rôle de filtre naturel pour les bassins versants.
   - Les **Cultures permanentes (`CP` — vignes, vergers, oliveraies, fruits à coque)** ne pèsent que **3,20 % de la surface nationale (`893 244 ha`)**, mais regroupent **7,83 % des parcelles (`765 642` parcelles)** en raison d'un parcellaire extrêmement morcelé.

2. **Forte asymétrie à droite et valeurs extrêmes de `surface_ha`** :
   - Sur l'ensemble du RPG, la surface médiane d'une parcelle est de **1,23 ha** (Q25 = **0,39 ha**, Q75 = **3,22 ha**), tandis que la moyenne s'établit à **2,86 ha** (soit plus du double de la médiane).
   - Par catégorie, les **Terres arables** ont les parcelles les plus vastes en médiane (**2,00 ha**, Q75 = **4,50 ha**, Q95 = **12,37 ha**, moyenne = **3,58 ha**), devant les **Prairies permanentes** (médiane = **1,21 ha**, moyenne = **3,00 ha**, mais un maximum à **6 529,65 ha** porté par les vastes estives et alpages collectifs de montagne).
   - À l'inverse, les **Cultures permanentes** présentent une surface médiane de seulement **0,63 ha** (moyenne = **1,17 ha**), typique du morcellement foncier viticole et arboricole. On observe en outre sur le boxplot que les parcelles en **Bio** sont systématiquement un peu plus petites en médiane que leurs homologues conventionnelles dans les Terres arables et les Cultures permanentes.

3. **Un taux de conversion au Bio très contrasté selon la catégorie** :
   - Les **Cultures permanentes** affichent le taux de surface en bio le plus élevé (**21,84 %** de leur surface), suivies des **Prairies permanentes** (**14,16 %**), tandis que les **Terres arables** sont presque trois fois moins converties (**7,49 %**), reflétant la plus forte dépendance technique des grandes cultures annuelles aux intrants de synthèse.

---

## 2. Paysage cultural français : groupes de cultures (`libelle_groupe`) et cultures principales (`libelle_culture`)

Affinons maintenant l'analyse au niveau des **23 groupes de cultures (`libelle_groupe`)** et du **Top 15 des cultures détaillées (`libelle_culture`)** parmi les 144 codes cultures du RPG 2025.

In [ ]:
# Agrégation par groupe de cultures (libelle_groupe)
stats_groupe = (
    df.groupby("libelle_groupe", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_ha=("surface_ha", "sum"),
        categorie_dominante=("libelle_categorie_propre", lambda s: s.mode().iloc[0] if not s.mode().empty else "Autre"),
    )
    .sort_values("surface_ha", ascending=False)
)
stats_groupe["surface_mha"] = stats_groupe["surface_ha"] / 1e6
stats_groupe["part_sau_%"] = 100 * stats_groupe["surface_ha"] / df["surface_ha"].sum()

# Agrégation pour le Top 15 des cultures détaillées (libelle_culture) avec distinction Conv. / Bio
top15_noms = (
    df.groupby("libelle_culture")["surface_ha"]
    .sum()
    .nlargest(15)
    .index
)
df_top15 = df[df["libelle_culture"].isin(top15_noms)]
pivot_cult = (
    df_top15.groupby(["libelle_culture", "bio"])["surface_ha"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(top15_noms)
) / 1e6

# Raccourcissement lisible des libellés très longs pour l'axe Y
labels_courts = [
    lbl[:52] + "…" if len(lbl) > 53 else lbl
    for lbl in pivot_cult.index
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 8.5))

# Sous-graphique 1 : Les 23 groupes de cultures colorés par grande catégorie dominante
palette_cat = {
    "Terres arables": "#2b8cbe",
    "Prairies permanentes": "#31a354",
    "Cultures permanentes": "#8856a7",
    "Non attribué / Hors production (IAE, SNE)": "#7f7f7f",
}
couleurs_barres_grp = [palette_cat.get(c, "#7f7f7f") for c in stats_groupe["categorie_dominante"]]

bars_grp = ax1.barh(
    stats_groupe["libelle_groupe"],
    stats_groupe["surface_mha"],
    color=couleurs_barres_grp,
    edgecolor="none",
    height=0.75,
)
ax1.invert_yaxis()
ax1.set_xlabel("Surface déclarée (millions d'hectares)")
ax1.set_xlim(0, 9.5)
ax1.set_title("Surface nationale des 23 groupes de cultures (`libelle_groupe`)", fontweight="bold", pad=12)

for bar, pct in zip(bars_grp, stats_groupe["part_sau_%"]):
    w = bar.get_width()
    ax1.text(
        w + 0.08,
        bar.get_y() + bar.get_height() / 2,
        f"{w:.2f} M ha ({pct:.1f} %)",
        va="center",
        fontsize=8.5,
    )

handles_cat = [mpatches.Patch(color=col, label=cat) for cat, col in palette_cat.items()]
ax1.legend(handles=handles_cat, title="Catégorie dominante", loc="lower right", frameon=True, facecolor="white", fontsize=8.5)

# Sous-graphique 2 : Top 15 des cultures détaillées (libelle_culture) empilé Conventionnel vs Bio
y_cult = np.arange(len(pivot_cult))
conv_mha = pivot_cult[False].to_numpy() if False in pivot_cult.columns else np.zeros(len(pivot_cult))
bio_mha = pivot_cult[True].to_numpy() if True in pivot_cult.columns else np.zeros(len(pivot_cult))
tot_mha = conv_mha + bio_mha

ax2.barh(y_cult, conv_mha, color="#6baed6", label="Conventionnel (`bio = False`)", height=0.72)
ax2.barh(y_cult, bio_mha, left=conv_mha, color="#238b45", label="Agriculture Biologique (`bio = True`)", height=0.72)

ax2.set_yticks(y_cult)
ax2.set_yticklabels(labels_courts, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel("Surface déclarée (millions d'hectares)")
ax2.set_xlim(0, 9.8)
ax2.set_title("Top 15 des cultures détaillées (`libelle_culture` sur 144)", fontweight="bold", pad=12)
ax2.legend(loc="center right", frameon=True, facecolor="white")

for idx, (tot, b_val) in enumerate(zip(tot_mha, bio_mha)):
    pct_b = 100 * b_val / tot if tot > 0 else 0
    ax2.text(
        tot + 0.08,
        idx,
        f"{tot:.2f} M ha (dont {pct_b:.1f} % bio)",
        va="center",
        fontsize=8.5,
    )

plt.tight_layout()
plt.show()

### Analyse de la spécialisation culturale française

Les deux graphiques ci-dessus révèlent une **concentration extrême de la SAU française** autour de quelques grandes productions :

1. **La hiérarchie des groupes de cultures (`libelle_groupe`)** :
   - Les **Prairies permanentes** arrivent largement en tête avec **7,69 millions d'ha** (**27,5 %** de la SAU, `3,03 millions` de parcelles), auxquelles s'ajoutent **2,33 millions d'ha** d'**Estives et landes** (**8,3 %**) et **1,40 million d'ha** de **Prairies temporaires** (**5,0 %**). Au total, les surfaces herbagères et pastorales couvrent plus de **40 % du territoire agricole**.
   - Du côté des grandes cultures de vente, trois groupes dominent : le **Blé tendre** (**4,46 M ha**, **16,0 %** de la SAU), le **Maïs grain et ensilage** (**2,81 M ha**, **10,1 %**) et l'**Orge** (**1,79 M ha**, **6,4 %**), suivis par les oléagineux que sont le **Colza** (**1,26 M ha**, **4,5 %**) et le **Tournesol** (**0,68 M ha**, **2,4 %**).
   - Les **Autres cultures industrielles** (qui regroupent les codes `24` et `25` : *Betterave* `419 457 ha` et *Pomme de terre* `236 096 ha`) totalisent **0,87 M ha** (**3,1 %**), et les **Vignes** occupent **0,58 M ha** (**2,1 %** de la SAU mais plus de `511 000` parcelles).

2. **Lecture croisée pour l'exposition aux produits phytosanitaires (BNV-D)** :
   - Sur les **144 codes cultures**, les **15 premières cultures représentent à elles seules 87,4 % de la SAU française** (`24,4 M ha`).
   - Or, l'usage des molécules étudiées dans notre contrat de données (`molecules.csv` : herbicides céréaliers/maïs comme le prosulfocarbe, le S-métolachlore ou le glyphosate, et fongicides viticoles/arboricoles) est directement gouverné par le ratio local entre :
     - les cultures à **Indice de Fréquence de Traitement (IFT) élevé** (*Blé tendre d'hiver* `4,43 M ha`, *Maïs* `2,79 M ha`, *Colza d'hiver* `1,26 M ha`, *Orge d'hiver et de printemps* `1,79 M ha`, *Vigne* `0,58 M ha`, *Betterave* `0,42 M ha`, *Pomme de terre* `0,24 M ha`) ;
     - et les surfaces **peu ou non traitées** (*Prairies permanentes `PPH`*, *Estives `SPH`/`SPL`*, *Jachères `JAC`* `0,50 M ha`, *Luzerne `LUZ`* `0,47 M ha`).

---

## 3. L'Agriculture Biologique (`bio`) selon les filières (`libelle_groupe`)

Dans quelles filières agricoles l'agriculture biologique est-elle la plus implantée en **part relative (%)**, et quelles filières apportent le plus grand **volume absolu d'hectares bio** en France ?

In [ ]:
# Calcul du taux de pénétration du bio et des surfaces bio par groupe de cultures
bio_groupe = (
    df.groupby("libelle_groupe", as_index=False)
    .agg(
        surface_totale_ha=("surface_ha", "sum"),
        surface_bio_ha=("surface_bio_ha", "sum"),
        nb_parcelles_tot=("surface_ha", "count"),
        nb_parcelles_bio=("bio", "sum"),
    )
)
bio_groupe["taux_bio_surface_%"] = 100 * bio_groupe["surface_bio_ha"] / bio_groupe["surface_totale_ha"]
bio_groupe["part_du_bio_national_%"] = 100 * bio_groupe["surface_bio_ha"] / bio_groupe["surface_bio_ha"].sum()

taux_national_bio = 100 * df["surface_bio_ha"].sum() / df["surface_ha"].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 8.2))

# Sous-graphique 1 : Taux de pénétration du bio (% au sein de chaque groupe)
bg_par_taux = bio_groupe.sort_values("taux_bio_surface_%", ascending=True)
couleurs_taux = ["#238b45" if v >= taux_national_bio else "#74c476" for v in bg_par_taux["taux_bio_surface_%"]]

bars1 = ax1.barh(bg_par_taux["libelle_groupe"], bg_par_taux["taux_bio_surface_%"], color=couleurs_taux, height=0.74)
ax1.axvline(taux_national_bio, color="#d7191c", linestyle="--", linewidth=1.6, label=f"Moyenne nationale ({taux_national_bio:.2f} %)")
ax1.set_xlabel("Part de la surface du groupe conduite en Bio (%)")
ax1.set_xlim(0, max(bg_par_taux["taux_bio_surface_%"]) * 1.18)
ax1.set_title("Taux de conversion au Bio au sein de chaque filière (%)", fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True, facecolor="white")

for bar in bars1:
    w = bar.get_width()
    ax1.text(w + 0.5, bar.get_y() + bar.get_height() / 2, f"{w:.1f} %", va="center", fontsize=8.5)

# Sous-graphique 2 : Contribution absolue au total des surfaces Bio françaises (en milliers d'ha)
bg_par_vol = bio_groupe.sort_values("surface_bio_ha", ascending=True)
bars2 = ax2.barh(bg_par_vol["libelle_groupe"], bg_par_vol["surface_bio_ha"] / 1e3, color="#2b8cbe", height=0.74)
ax2.set_xlabel("Surface Bio déclarée (en milliers d'hectares)")
ax2.set_xlim(0, (bg_par_vol["surface_bio_ha"].max() / 1e3) * 1.25)
ax2.set_title("Répartition des 2,90 millions d'ha Bio par groupe de cultures", fontweight="bold", pad=12)

for bar, part_nat in zip(bars2, bg_par_vol["part_du_bio_national_%"]):
    w = bar.get_width()
    ax2.text(
        w + 12,
        bar.get_y() + bar.get_height() / 2,
        f"{w:,.0f} k ha ({part_nat:.1f} % du bio)".replace(",", " "),
        va="center",
        fontsize=8.5,
    )

plt.tight_layout()
plt.show()

### Analyse de la diffusion de l'Agriculture Biologique par filière

La confrontation du **taux de conversion interne à chaque filière** (à gauche) et du **poids en surface absolue dans le bio français** (à droite) met en évidence une dualité fondamentale :

1. **Des taux de conversion records dans les cultures permanentes, les légumineuses et les fourrages** :
   - Les filières arboricoles, protéagineuses et méditerranéennes affichent les taux de conversion les plus élevés de France : **Oliviers (47,1 %)**, **Légumineuses à grains (36,3 %)**, **Fruits à coque (33,7 %)**, **Fourrage (32,1 %)**, **Estives et landes (26,0 %)**, **Vergers (24,5 %)**, **Autres oléagineux (22,1 % — soja, lin oléagineux)**, **Protéagineux (21,3 %)** et **Vignes (21,3 %)**.
   - Ce phénomène s'explique à la fois par une forte valorisation économique du label bio en filières spécialisées (vin bio, fruits, huile d'olive, lentilles/pois chiches) et par le rôle agronomique central des légumineuses et fourrages (*luzerne* à **32,0 %** en bio, *mélanges légumineuses/graminées* à **25,6 %** en bio), indispensables dans les rotations biologiques pour fixer l'azote atmosphérique sans engrais de synthèse.

2. **Une très faible pénétration du Bio dans les grandes cultures céréalières et industrielles** :
   - À l'opposé du spectre, les grandes cultures de plaine restent très majoritairement conventionnelles : le **Blé tendre (2,8 %)**, les **Plantes à fibres (2,7 %)**, l'**Orge (2,7 %)**, le **Maïs grain et ensilage (2,0 %)** et surtout le **Colza (0,8 %)** ou la **Betterave (0,7 %)** présentent des taux de bio très inférieurs à la moyenne nationale de **10,38 %**.
   - En termes de **volumes absolus** (graphique de droite), ce sont donc les **surfaces herbagères et fourragères** qui fournissent plus des **deux tiers (69,5 %) des 2,90 millions d'hectares bio** français : **Prairies permanentes** (`810 k ha`, **27,9 %** du bio national), **Estives et landes** (`606 k ha`, **20,9 %**), **Fourrage** (`391 k ha`, **13,5 %**) et **Prairies temporaires** (`208 k ha`, **7,2 %**), loin devant les **Autres céréales** (`137 k ha`), les **Vignes** (`123 k ha`) et le **Blé tendre** (`123 k ha`).

3. **Conséquence directe pour notre étude *Eau × Pesticides*** :
   - Un département ayant un fort taux de SAU bio parce qu'il est dominé par les prairies ou les estives n'a pas la même exposition de base aux herbicides céréaliers qu'un département de grande culture. Lors de l'analyse croisée avec Hub'Eau, il sera donc essentiel de contrôler simultanément par la **structure des groupes de cultures (`libelle_groupe`)** et par le **taux de bio (`bio`)**.

---

## 4. Disparités régionales : ancrage territorial du Bio et spécialisation des régions

Comment ces logiques de filières se traduisent-elles géographiquement entre les **13 régions métropolitaines et l'Outre-Mer** ? Nous croisons ici le taux régional de surface bio avec le profil cultural de chaque région.

In [ ]:
# Regroupement des 23 groupes de cultures en 6 grandes familles agronomiques lisibles pour le profil régional
FAMILLE_AGRONOMIQUE = {
    "Blé tendre": "Céréales à paille & Riz",
    "Orge": "Céréales à paille & Riz",
    "Autres céréales": "Céréales à paille & Riz",
    "Riz": "Céréales à paille & Riz",
    "Maïs grain et ensilage": "Maïs (grain & ensilage)",
    "Colza": "Oléagineux, Protéagineux & Indus.",
    "Tournesol": "Oléagineux, Protéagineux & Indus.",
    "Autres oléagineux": "Oléagineux, Protéagineux & Indus.",
    "Protéagineux": "Oléagineux, Protéagineux & Indus.",
    "Légumineuses à grains": "Oléagineux, Protéagineux & Indus.",
    "Plantes à fibres": "Oléagineux, Protéagineux & Indus.",
    "Autres cultures industrielles": "Oléagineux, Protéagineux & Indus.",
    "Canne à sucre": "Oléagineux, Protéagineux & Indus.",
    "Prairies permanentes": "Prairies permanentes & Estives",
    "Estives et landes": "Prairies permanentes & Estives",
    "Prairies temporaires": "Prairies temp. & Fourrages",
    "Fourrage": "Prairies temp. & Fourrages",
    "Vignes": "Vignes, Vergers & Cultures perm.",
    "Vergers": "Vignes, Vergers & Cultures perm.",
    "Fruits à coque": "Vignes, Vergers & Cultures perm.",
    "Oliviers": "Vignes, Vergers & Cultures perm.",
    "Gel (surfaces gelées sans production)": "Jachères & Divers",
    "Divers": "Jachères & Divers",
}
df["famille_agro"] = df["libelle_groupe"].map(FAMILLE_AGRONOMIQUE).fillna("Jachères & Divers")

# Statistiques régionales Bio
stats_region = (
    df.groupby("region", as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        surface_bio_ha=("surface_bio_ha", "sum"),
        surface_moy_parcelle_ha=("surface_ha", "mean"),
    )
)
stats_region["taux_bio_%"] = 100 * stats_region["surface_bio_ha"] / stats_region["surface_totale_ha"]
stats_region = stats_region.sort_values("taux_bio_%", ascending=True)

# Profil cultural 100 % empilé par région (ordonné comme stats_region)
ordre_familles = [
    "Prairies permanentes & Estives",
    "Prairies temp. & Fourrages",
    "Vignes, Vergers & Cultures perm.",
    "Maïs (grain & ensilage)",
    "Céréales à paille & Riz",
    "Oléagineux, Protéagineux & Indus.",
    "Jachères & Divers",
]
couleurs_familles = {
    "Prairies permanentes & Estives": "#31a354",
    "Prairies temp. & Fourrages": "#a1d99b",
    "Vignes, Vergers & Cultures perm.": "#8856a7",
    "Maïs (grain & ensilage)": "#fd8d3c",
    "Céréales à paille & Riz": "#2b8cbe",
    "Oléagineux, Protéagineux & Indus.": "#feb24c",
    "Jachères & Divers": "#bdbdbd",
}

piv_reg_fam = (
    df.groupby(["region", "famille_agro"])["surface_ha"]
    .sum()
    .unstack(fill_value=0.0)
    .reindex(index=stats_region["region"], columns=ordre_familles, fill_value=0.0)
)
piv_reg_pct = piv_reg_fam.div(piv_reg_fam.sum(axis=1), axis=0) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 7.2), sharey=True)

# Sous-graphique 1 : Taux de surface Bio par région
couleurs_reg = ["#006d2c" if v >= taux_national_bio else "#6baed6" for v in stats_region["taux_bio_%"]]
bars_r = ax1.barh(stats_region["region"], stats_region["taux_bio_%"], color=couleurs_reg, height=0.72)
ax1.axvline(taux_national_bio, color="#d7191c", linestyle="--", linewidth=1.6, label=f"Moyenne France ({taux_national_bio:.2f} %)")
ax1.set_xlabel("Part de la SAU régionale en Agriculture Biologique (%)")
ax1.set_xlim(0, max(stats_region["taux_bio_%"]) * 1.25)
ax1.set_title("Ancrage régional de l'Agriculture Biologique (% SAU)", fontweight="bold", pad=12)
ax1.legend(loc="lower right", frameon=True, facecolor="white")

for bar, surf_b in zip(bars_r, stats_region["surface_bio_ha"]):
    w = bar.get_width()
    ax1.text(
        w + 0.4,
        bar.get_y() + bar.get_height() / 2,
        f"{w:.1f} % ({surf_b / 1e3:,.0f} k ha)".replace(",", " "),
        va="center",
        fontsize=8.5,
    )

# Sous-graphique 2 : Profil cultural 100 % empilé de chaque région
cumul = np.zeros(len(piv_reg_pct))
for fam in ordre_familles:
    vals = piv_reg_pct[fam].to_numpy()
    ax2.barh(
        piv_reg_pct.index,
        vals,
        left=cumul,
        color=couleurs_familles[fam],
        edgecolor="white",
        linewidth=0.4,
        height=0.72,
        label=fam,
    )
    cumul += vals

ax2.set_xlim(0, 100)
ax2.set_xlabel("Part dans la SAU régionale (%)")
ax2.set_title("Spécialisation culturale des régions (% de la SAU régionale)", fontweight="bold", pad=12)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3, frameon=True, facecolor="white", fontsize=8.5)

plt.tight_layout()
plt.show()

### Analyse des disparités régionales : le clivage Nord-Sud et Grandes Plaines vs Montagne/Littoral

La mise en regard directe du **taux régional d'agriculture biologique** (à gauche) et de la **composition culturale** (à droite) éclaire les grands équilibres territoriaux français :

1. **Une fracture Nord / Sud très marquée sur l'Agriculture Biologique** :
   - Quatre régions dépassent la moyenne nationale (**10,38 %**) : **Provence-Alpes-Côte d'Azur** (**24,7 %** de sa SAU en bio, soit `259 k ha`), **Occitanie** (**22,5 %**, et **1er bassin national en volume** avec `825 k ha` bio), **Corse** (**14,4 %**, `44 k ha`) et **Pays de la Loire** (**12,0 %**, `246 k ha`), suivies de près par **Auvergne-Rhône-Alpes** (**10,1 %**, `309 k ha`), la **Bretagne** (**9,8 %**, `156 k ha`), la **Bourgogne-Franche-Comté** (**9,5 %**, `228 k ha`) et la **Nouvelle-Aquitaine** (**8,1 %**, `313 k ha`).
   - À l'inverse, les grandes plaines du Nord et du Bassin parisien affichent les parts de SAU bio les plus faibles : **Hauts-de-France (2,4 %**, `50 k ha`), **Centre-Val de Loire (4,5 %**, `101 k ha`), **Normandie (6,2 %**, `118 k ha`), **Île-de-France (6,7 %**, `37 k ha`) et **Grand Est (6,9 %**, `206 k ha`).

2. **Comment la spécialisation régionale explique la géographie des pressions agricoles** :
   - En **Île-de-France**, **Centre-Val de Loire**, **Hauts-de-France** et **Grand Est**, le profil cultural (à droite) est dominé à plus de **60–85 % par le bloc Céréales + Oléagineux/Protéagineux/Cultures industrielles + Maïs**. Ce sont des systèmes d'openfield à haute productivité, très consommateurs d'herbicides et de fongicides, où la conversion au bio reste faible.
   - En **Bretagne** et **Pays de la Loire**, on observe le profil typique de la polyculture-élevage de l'Ouest : forte part de **Maïs (grain et ensilage)** et de **Prairies temporaires & Fourrages** en rotation avec les céréales, mais avec une dynamique bio nettement plus marquée (9,8 % à 12,0 %).
   - En **Corse**, **PACA**, **Auvergne-Rhône-Alpes** et **Occitanie**, la combinaison des **Prairies permanentes & Estives** et des **Vignes, Vergers & Cultures permanentes** soutient des taux de bio supérieurs à 10–24 %.

---

## 5. Cartographie départementale (Choroplèthes) et carte parcellaire nationale

Pour préparer la jointure à la **maille départementale (`code_dep`)** prévue par notre [`contrat_donnees.md`](../docs/contrat_donnees.md), nous représentons trois cartes choroplèthes complémentaires sur les 96 départements métropolitains, suivies de la carte nationale des **9,78 millions de parcelles**.

In [ ]:
# Agrégation par département (maille de jointure du contrat de données)
GROUPES_FORTE_PRESSION = {
    "Blé tendre", "Maïs grain et ensilage", "Orge", "Autres céréales",
    "Colza", "Tournesol", "Autres oléagineux", "Protéagineux",
    "Plantes à fibres", "Autres cultures industrielles",
    "Vignes", "Vergers", "Fruits à coque", "Oliviers",
}
df["culture_pression_phyto"] = df["libelle_groupe"].isin(GROUPES_FORTE_PRESSION)
df["surface_pression_ha"] = np.where(df["culture_pression_phyto"], df["surface_ha"], 0.0)

stats_dep = (
    df.groupby(["code_dep", "nom_dep"], as_index=False)
    .agg(
        nb_parcelles=("surface_ha", "count"),
        surface_totale_ha=("surface_ha", "sum"),
        surface_moy_ha=("surface_ha", "mean"),
        surface_bio_ha=("surface_bio_ha", "sum"),
        surface_pression_ha=("surface_pression_ha", "sum"),
    )
)
stats_dep["taux_bio_%"] = 100 * stats_dep["surface_bio_ha"] / stats_dep["surface_totale_ha"]
stats_dep["part_cultures_intensives_%"] = 100 * stats_dep["surface_pression_ha"] / stats_dep["surface_totale_ha"]

# Chargement des contours départementaux mis en cache (France métropolitaine en Lambert-93)
chemin_contours = Path("data/processed/rpg/_contours_departements.parquet")
if chemin_contours.exists():
    deps_gdf = gpd.read_parquet(chemin_contours)
else:
    from src.visualisation.carte_parcelles_rpg import _charger_contours_departements
    deps_gdf = _charger_contours_departements()

deps_metro = deps_gdf[~deps_gdf["code_dep"].isin({"971", "972", "973", "974", "976"})].to_crs(2154)
carte_dep = deps_metro.merge(stats_dep, on="code_dep", how="left")

# Tracé des 3 cartes choroplèthes départementales côte à côte (bornes ajustées hors micro-département 75 = 3 parcelles)
fig, axes = plt.subplots(1, 3, figsize=(19, 6.8))

configurations_cartes = [
    (
        "part_cultures_intensives_%",
        "YlOrRd",
        0.0,
        95.0,
        "1. Part des grandes cultures, vignes & vergers\ndans la SAU départementale (%)",
        "% de la SAU",
    ),
    (
        "taux_bio_%",
        "YlGn",
        0.0,
        36.0,
        "2. Part de l'Agriculture Biologique (`bio = True`)\ndans la SAU départementale (%)",
        "% de la SAU en Bio (plafonné à 36 %)",
    ),
    (
        "surface_moy_ha",
        "PuBu",
        0.8,
        7.0,
        "3. Surface moyenne d'une parcelle (`surface_ha`)\npar département (en ha)",
        "Surface moyenne en ha (plafonnée à 7 ha)",
    ),
]

for ax, (col, cmap, vmin, vmax, titre, label_cb) in zip(axes, configurations_cartes):
    carte_dep.plot(
        column=col,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        linewidth=0.45,
        edgecolor="#333333",
        legend=True,
        legend_kwds={"label": label_cb, "orientation": "horizontal", "shrink": 0.78, "pad": 0.03},
        ax=ax,
    )
    ax.set_title(titre, fontweight="bold", fontsize=11, pad=10)
    ax.set_axis_off()

plt.tight_layout()
plt.show()

# Affichage de la carte parcellaire nationale (9,78 millions de parcelles) générée par src/visualisation/carte_parcelles_rpg.py
chemin_carte_france = Path("data/processed/carte_rpg_2025_france_groupes.png")
if chemin_carte_france.exists():
    display(Image(filename=str(chemin_carte_france), width=1050))

### Analyse spatiale des cartes départementales et de la carte parcellaire

Les trois cartes choroplèthes départementales et la carte parcellaire haute résolution mettent en évidence une structuration spatiale très nette :
- **Carte 1 (Part des grandes cultures, vignes et vergers)** : Le grand arc céréalier et industriel (**Beauce, Brie, Picardie, Champagne crayeuse, Poitou-Charentes, plaine d'Alsace, vallée de la Garonne**) concentre plus de **75 à 90 % de sa SAU** dans des cultures à forte pression phytosanitaire, en opposition complète avec la diagonale herbagère et montagnarde (**Massif Central, Jura, Vosges, Alpes, Pyrénées, bocage normand**).
- **Carte 2 (Ancrage départemental du Bio)** : La part de SAU en bio présente une image presque inversée de la carte 1, avec un maximum dans le quart Sud-Est et le Sud-Ouest (**Drôme, Hautes-Alpes, Alpes-de-Haute-Provence, Bouches-du-Rhône, Var, Corse, Lozère, Gers, Ariège**) où elle atteint **20 à 36 % de la SAU**, contre moins de **2 à 4 %** dans l'Aisne, la Somme, le Pas-de-Calais, l'Oise ou la Marne.
- **Carte 3 (Surface moyenne des parcelles)** : On retrouve l'empreinte conjointe de l'openfield du Bassin parisien et du Berry (parcelles moyennes de **5 à 7 ha**) ainsi que des grands alpages/estives des Alpes du Sud, de la Corse et des Pyrénées, face au parcellaire très fin (< **2 ha** en moyenne) de la Bretagne, de la façade atlantique et des bassins viticoles.

---

### Q&A
- **Comment se structurent les surfaces agricoles du RPG 2025 selon les catégories, groupes, cultures et le mode biologique ?**
  Le RPG 2025 totalise **9 776 236 parcelles** pour **27 924 774 ha** déclarés sur 101 départements. L'espace agricole est dominé par les **Terres arables** (**16,68 M ha**, soit **59,73 %** de la SAU, **7,49 %** en bio) et les **Prairies permanentes** (**10,08 M ha**, **36,10 %** de la SAU, **14,16 %** en bio), tandis que les **Cultures permanentes** (vignes, vergers, oliveraies) représentent **0,89 M ha** (**3,20 %** de la SAU mais **7,83 %** des parcelles et **21,84 %** en bio). L'agriculture biologique (`bio = True`) couvre au total **2 899 677 ha** (**10,38 %** de la SAU et **11,41 %** des parcelles), avec de très forts contrastes entre filières et entre régions.

### Data Analysis Key Findings
- **Asymétrie marquée de la taille des parcelles (`surface_ha`)** : La surface médiane d'une parcelle française est de **1,23 ha** (Q25 = **0,39 ha**, Q75 = **3,22 ha**) pour une moyenne de **2,86 ha** et un 95e centile à **10,49 ha** (médiane de **2,00 ha** en Terres arables, **1,21 ha** en Prairies permanentes et **0,63 ha** en Cultures permanentes).
- **Forte concentration culturale (`libelle_groupe` et `libelle_culture`)** : Sur 144 cultures détaillées, les **15 premières concentrent 87,4 % de la SAU**. Trois groupes rassemblent plus de la moitié des terres : **Prairies permanentes** (**7,69 M ha**, 27,5 %), **Blé tendre** (**4,46 M ha**, 16,0 %) et **Maïs grain et ensilage** (**2,81 M ha**, 10,1 %), suivis des **Estives et landes** (**2,33 M ha**), de l'**Orge** (**1,79 M ha**), des **Prairies temporaires** (**1,40 M ha**) et du **Colza** (**1,26 M ha**).
- **Dualité du Bio entre taux de conversion et volume d'hectares** : Si les filières spécialisées et méditerranéennes (*Oliviers* **47,1 %**, *Légumineuses à grains* **36,3 %**, *Fruits à coque* **33,7 %**, *Fourrage* **32,1 %**, *Vergers* **24,5 %**, *Vignes* **21,3 %**) affichent les taux de conversion internes les plus élevés, ce sont les surfaces herbagères et fourragères qui apportent **69,5 % des 2,90 M ha bio** (`810 k ha` de prairies permanentes, `606 k ha` d'estives, `391 k ha` de fourrages, `208 k ha` de prairies temporaires), tandis que le blé tendre (**2,8 %**), l'orge (**2,7 %**), le maïs (**2,0 %**), le colza (**0,8 %**) et la betterave (**0,7 %**) restent très peu convertis.
- **Clivage territorial net** : Les régions du Sud et de l'Ouest (**PACA 24,7 %**, **Occitanie 22,5 %** soit `825 k ha`, **Corse 14,4 %**, **Pays de la Loire 12,0 %**) devancent fortement les plaines de grandes cultures du Nord et du Centre (**Hauts-de-France 2,4 %**, **Centre-Val de Loire 4,5 %**, **Normandie 6,2 %**, **Île-de-France 6,7 %**, **Grand Est 6,9 %**).

### Insights or Next Steps
- **Création d'indicateurs explicatifs pour la jointure `base_finale.parquet`** : Lors du croisement avec les achats de pesticides (**BNV-D**) et les analyses d'eau (**Hub'Eau**), ne pas se limiter à la surface agricole totale par département : inclure la **part départementale de cultures à fort IFT** (*céréales, colza, maïs, betterave, pomme de terre, vigne*) ainsi que le **taux de surface en bio (`taux_bio_%`)** comme variable modératrice de la pression phytosanitaire.
- **Construction du livrable [`data/processed/champs.parquet`](../docs/contrat_donnees.md)** : Agréger les parcelles à la maille `code_dep × annee × code_groupe_culture` conformément au contrat de données du groupe afin de réaliser la fusion avec `eau.parquet` et `pesticides.parquet`.